# Practice 103 -- Difference-in-Differences & the Staggered Adoption Problem

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module --
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_staggered_panel, load_2x2_subset, load_event_study_subset, to_differences_cohort_column
from src.plotting import event_study_plot, bacon_decomposition_plot, method_comparison_plot

panel = load_staggered_panel(seed=0)
df = panel.df
df.head()

## Phase 1 -- The canonical 2x2 DiD

Before anything staggered: one treated cohort (`"mid"`, first treated at `t=5`), one
never-treated control, one pre period (`t=4`), one post period (`t=5`). The double
difference is the whole estimator -- everything later in this notebook is many copies
of this exact computation.

In [ ]:
sub_2x2 = load_2x2_subset(panel, cohort="mid", control="never")
sub_2x2.head()

### Exercise -- `src/_01_two_by_two_did.py :: did_2x2`

Open `src/_01_two_by_two_did.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_two_by_two_did import did_2x2, compare_to_regression

compare_to_regression(sub_2x2)
result = did_2x2(sub_2x2)
result

## Phase 2 -- The event-study specification & testing pre-trends

Replace the single `post` dummy with one dummy per relative-time bin, so every period
in the panel gets its own coefficient. The *pre*-period coefficients are the standard
pre-trends check -- they should be indistinguishable from zero if parallel trends holds.

In [ ]:
from src._02_event_study import fit_event_study_model

event_df = load_event_study_subset(panel, cohort="mid", control="never")
model = fit_event_study_model(event_df)
model.summary()

### Exercise -- `src/_02_event_study.py :: event_study_coefficients`

Open `src/_02_event_study.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._02_event_study import event_study_coefficients

es_table = event_study_coefficients(model)
print(es_table.to_string(index=False))
fig = event_study_plot(es_table, title="Phase 2 -- 'mid' cohort vs. never-treated")
fig

## Phase 3 -- Staggered adoption breaks two-way fixed effects

Now the full panel: three cohorts adopting at different times (`early`, `mid`, `late`),
plus a never-treated group. `early`'s treatment effect keeps growing the longer it's
been treated; `mid` and `late` have flat effects. Every true effect in the data is
**positive** -- watch what the naive static TWFE regression does anyway.

In [ ]:
from src._03_twfe_and_bacon import fit_naive_twfe

true_min = panel.true_att_by_cell["true_att"].min()
true_max = panel.true_att_by_cell["true_att"].max()
twfe = fit_naive_twfe(df)
print(f"True effect range across every (g, t) cell: [{true_min:.2f}, {true_max:.2f}]")
print(f"Naive TWFE estimate: {twfe.coef()['treated']:.3f} (se={twfe.se()['treated']:.3f})")

### Exercise -- `src/_03_twfe_and_bacon.py :: goodman_bacon_decompose`

Open `src/_03_twfe_and_bacon.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below. It requires Phase 1's `did_2x2`.

In [ ]:
from src._03_twfe_and_bacon import goodman_bacon_decompose

bacon = goodman_bacon_decompose(df)
print(bacon.to_string(index=False))
fig = bacon_decomposition_plot(bacon, title="Phase 3 -- pairwise 2x2s behind the TWFE coefficient")
fig

## Phase 4 -- The repair: Callaway-Sant'Anna group-time ATT(g, t)

Instead of one pooled regression, estimate one clean 2x2 per (cohort, period) cell --
always against the never-treated group, so no "forbidden" comparison can sneak in.

### Exercise -- `src/_04_att_gt.py :: att_gt`

Open `src/_04_att_gt.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below. It also requires Phase 1's `did_2x2`.

In [ ]:
from src._04_att_gt import compute_all_att_gt

att_gt_df = compute_all_att_gt(df)
checked = att_gt_df.merge(panel.true_att_by_cell, on=["g", "t"])
checked["abs_error"] = (checked["att"] - checked["true_att"]).abs()
print(checked.to_string(index=False))
print(f"\nMax |att_hat - att_true|: {checked['abs_error'].max():.3f}")

## Phase 5 -- Aggregating ATT(g, t)

Two aggregations of the same table: an event-study path (one number per relative time)
and a single overall ATT -- the two numbers a Callaway-Sant'Anna paper actually reports.

### Exercise -- `src/_05_aggregate_att.py :: aggregate_event_study`, `aggregate_overall_att`

Open `src/_05_aggregate_att.py`, read the `TODO(human)` block above the two functions,
implement them there, then re-run the cell below.

In [ ]:
from src._05_aggregate_att import aggregate_event_study, aggregate_overall_att

es_agg = aggregate_event_study(att_gt_df)
print(es_agg.to_string(index=False))
fig = event_study_plot(es_agg.rename(columns={"att": "estimate"}), title="Phase 5 -- aggregated event study (all cohorts)")
fig

In [ ]:
overall_att, overall_se = aggregate_overall_att(att_gt_df)
print(f"Overall Callaway-Sant'Anna ATT: {overall_att:.3f} (se={overall_se:.3f})")
print(f"Naive TWFE:                     {twfe.coef()['treated']:.3f}")

## End-to-end: TWFE vs. Callaway-Sant'Anna vs. truth, and a cross-check against `differences`

The `differences` package implements Callaway-Sant'Anna independently -- if the hand-rolled
ATT(g,t) above is correct, it should land close to what `differences` reports on the exact
same panel. Python's staggered-DiD tooling is younger and less battle-tested than R's, which
is exactly why this practice built the estimator by hand instead of trusting a one-line call.

In [ ]:
import differences

true_overall_att = panel.true_att_by_cell["true_att"].mean()

diff_df = df.copy()
diff_df["cohort_col"] = to_differences_cohort_column(diff_df["g"])
diff_df = diff_df.set_index(["unit", "time"])
att_model = differences.ATTgt(data=diff_df, cohort_column="cohort_col")
diff_result = att_model.fit(formula="y ~ 1", control_group="never_treated")
diff_overall = diff_result.aggregate("simple")
diff_att = float(diff_overall[("SimpleAggregation", "", "ATT")].iloc[0])
diff_se = float(diff_overall[("SimpleAggregation", "analytic", "std_error")].iloc[0])
print(f"differences (CS) overall ATT: {diff_att:.3f} (se={diff_se:.3f})")

In [ ]:
fig = method_comparison_plot(
    ["TWFE", "CS (hand-rolled)", "differences (CS)"],
    [twfe.coef()["treated"], overall_att, diff_att],
    [twfe.se()["treated"], overall_se, diff_se],
    true_att=true_overall_att,
    title="TWFE vs. Callaway-Sant'Anna vs. truth",
)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert abs(result.estimate - 1.0) < 0.3, "Phase 1: 2x2 DiD should be close to the true 'mid' effect (1.0)"
assert es_table.loc[es_table.rel_time < 0, "estimate"].abs().max() < 0.5, "Phase 2: pre-trend coefficients should be small"
assert not (true_min <= twfe.coef()["treated"] <= true_max), "Phase 3: naive TWFE should fall outside the true effect range"
assert bacon.loc[bacon.type == "forbidden", "estimate"].min() < 0, "Phase 3: at least one forbidden comparison should be negative"
assert checked["abs_error"].max() < 1.0, "Phase 4: ATT(g,t) should closely match the ground truth"
assert abs(overall_att - true_overall_att) < 1.0, "Phase 5: aggregated CS ATT should be close to the true overall ATT"
print("OK")